### **Installations, Imports and Configurations**

In [1]:
# ============================================================
# Cell 1 — Installations for Qwen3-4B LoRA + ntkmirror
# ============================================================
#
# Goal:
#   Use an already fine-tuned Qwen3-4B LoRA adapter as the frozen model,
#   then train an ntkmirror controller on top of it.
#
# This cell installs only the packages needed for:
#   - loading Qwen3-4B in 4-bit
#   - loading a PEFT/LoRA adapter
#   - training/evaluating ntkmirror
#   - computing BLEU, spBLEU, chrF, chrF++
#   - later computing E5 semantic similarity
# ============================================================

import importlib.util
import subprocess
import sys
from importlib.metadata import version as pkg_version, PackageNotFoundError


def is_installed(import_name: str) -> bool:
    return importlib.util.find_spec(import_name) is not None


def installed_version(package_name: str):
    try:
        return pkg_version(package_name)
    except PackageNotFoundError:
        return None


# package_name_on_pip : import_name_in_python
required = {
    "transformers": "transformers",
    "peft": "peft",
    "accelerate": "accelerate",
    "bitsandbytes": "bitsandbytes",
    "datasets": "datasets",
    "sacrebleu": "sacrebleu",
    "sentencepiece": "sentencepiece",
    "packaging": "packaging",
    "tqdm": "tqdm",
    "pandas": "pandas",
    "numpy": "numpy",
    "openpyxl": "openpyxl",
    "sentence-transformers": "sentence_transformers",
}

missing = [
    pip_name
    for pip_name, import_name in required.items()
    if not is_installed(import_name)
]

print("Missing packages:", missing)

if missing:
    cmd = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--no-cache-dir",
        *missing,
    ]
    print("Running:", " ".join(cmd))
    subprocess.check_call(cmd)
else:
    print("All required regular packages are already installed.")


# ------------------------------------------------------------
# Install ntkmirror from GitHub
# ------------------------------------------------------------

if not is_installed("ntkmirror"):
    cmd = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--no-cache-dir",
        "git+https://github.com/leochlon/ntkmirror.git",
    ]
    print("Installing ntkmirror:")
    print("Running:", " ".join(cmd))
    subprocess.check_call(cmd)
else:
    print("ntkmirror is already installed.")


# ------------------------------------------------------------
# Qwen3 requires a recent Transformers version
# ------------------------------------------------------------

from packaging.version import parse as parse_version

transformers_version = installed_version("transformers")
print("Current transformers:", transformers_version)

if transformers_version is None or parse_version(transformers_version) < parse_version("4.51.0"):
    cmd = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--upgrade",
        "transformers>=4.51.0",
    ]
    print("Upgrading transformers for Qwen3 support:")
    print("Running:", " ".join(cmd))
    subprocess.check_call(cmd)
else:
    print("transformers version is OK for Qwen3.")


print("\nCell 1 finished.")
print("If Colab asks for a runtime restart after installation, restart and rerun Cell 1.")

Missing packages: ['bitsandbytes', 'sacrebleu']
Running: /usr/bin/python3 -m pip install -q --no-cache-dir bitsandbytes sacrebleu
Installing ntkmirror:
Running: /usr/bin/python3 -m pip install -q --no-cache-dir git+https://github.com/leochlon/ntkmirror.git
Current transformers: 5.10.2
transformers version is OK for Qwen3.

Cell 1 finished.
If Colab asks for a runtime restart after installation, restart and rerun Cell 1.


In [2]:
# ============================================================
# Cell 1B — Environment check
# ============================================================
#
# Checks:
#   - Python / PyTorch / CUDA
#   - GPU name and VRAM
#   - important package versions
#   - ntkmirror import availability
#
# This notebook is expected to run on Colab T4-like hardware.
# ============================================================

import os
import sys
import random
import platform
import subprocess
from importlib.metadata import version as pkg_version, PackageNotFoundError

import numpy as np
import torch


# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Seed:", SEED)


# ------------------------------------------------------------
# Basic environment
# ------------------------------------------------------------

print("\n================ Python / System ================")
print("Python:", sys.version)
print("Platform:", platform.platform())
print("Executable:", sys.executable)


# ------------------------------------------------------------
# PyTorch / CUDA
# ------------------------------------------------------------

print("\n================ PyTorch / CUDA ================")
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    total_vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    reserved_gb = torch.cuda.memory_reserved(0) / 1024**3
    allocated_gb = torch.cuda.memory_allocated(0) / 1024**3

    print("GPU:", gpu_name)
    print(f"Total VRAM: {total_vram_gb:.2f} GB")
    print(f"Reserved VRAM: {reserved_gb:.2f} GB")
    print(f"Allocated VRAM: {allocated_gb:.2f} GB")
else:
    print("WARNING: No GPU detected. This notebook will be too slow without CUDA.")


# ------------------------------------------------------------
# nvidia-smi
# ------------------------------------------------------------

print("\n================ nvidia-smi ================")
try:
    result = subprocess.run(
        ["nvidia-smi"],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
    )
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
except Exception as e:
    print("Could not run nvidia-smi:", repr(e))


# ------------------------------------------------------------
# Package versions
# ------------------------------------------------------------

def safe_version(package_name):
    try:
        return pkg_version(package_name)
    except PackageNotFoundError:
        return "NOT INSTALLED"


packages_to_check = [
    "transformers",
    "peft",
    "accelerate",
    "bitsandbytes",
    "datasets",
    "sacrebleu",
    "sentence-transformers",
    "sentencepiece",
    "ntkmirror",
]

print("\n================ Package versions ================")
for pkg in packages_to_check:
    print(f"{pkg:24s}: {safe_version(pkg)}")


# ------------------------------------------------------------
# Import checks
# ------------------------------------------------------------

print("\n================ Import checks ================")

try:
    import transformers
    print("transformers import: OK")
except Exception as e:
    print("transformers import failed:", repr(e))

try:
    import peft
    print("peft import: OK")
except Exception as e:
    print("peft import failed:", repr(e))

try:
    import bitsandbytes as bnb
    print("bitsandbytes import: OK")
except Exception as e:
    print("bitsandbytes import failed:", repr(e))

try:
    import ntkmirror
    print("ntkmirror import: OK")
    print("ntkmirror module:", ntkmirror)
except Exception as e:
    print("ntkmirror import failed:", repr(e))


# ------------------------------------------------------------
# TF32 settings
# ------------------------------------------------------------

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    print("\nTF32 enabled for CUDA matmul/cudnn where supported.")


print("\nCell 1B finished.")

Seed: 42

================ Python / System ================
Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Platform: Linux-6.6.122+-x86_64-with-glibc2.35
Executable: /usr/bin/python3

================ PyTorch / CUDA ================
Torch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
Total VRAM: 14.56 GB
Reserved VRAM: 0.00 GB
Allocated VRAM: 0.00 GB

================ nvidia-smi ================
Wed Jun 17 04:50:12 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=

In [3]:
# ============================================================
# Cell 2 — Mount Google Drive and define project paths
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/alexandria_qwen35_sft")

DATA_DIR = PROJECT_DIR / "prepared_data"
RUNS_DIR = PROJECT_DIR / "runs"
ADAPTER_DIR = PROJECT_DIR / "final_adapters"
PRED_DIR = PROJECT_DIR / "predictions"
SEMANTIC_DIR = PROJECT_DIR / "semantic_similarity_e5_large"

for p in [PROJECT_DIR, DATA_DIR, RUNS_DIR, ADAPTER_DIR, PRED_DIR, SEMANTIC_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("DATA_DIR:", DATA_DIR)
print("RUNS_DIR:", RUNS_DIR)
print("ADAPTER_DIR:", ADAPTER_DIR)
print("PRED_DIR:", PRED_DIR)
print("SEMANTIC_DIR:", SEMANTIC_DIR)

Mounted at /content/drive
PROJECT_DIR: /content/drive/MyDrive/alexandria_qwen35_sft
DATA_DIR: /content/drive/MyDrive/alexandria_qwen35_sft/prepared_data
RUNS_DIR: /content/drive/MyDrive/alexandria_qwen35_sft/runs
ADAPTER_DIR: /content/drive/MyDrive/alexandria_qwen35_sft/final_adapters
PRED_DIR: /content/drive/MyDrive/alexandria_qwen35_sft/predictions
SEMANTIC_DIR: /content/drive/MyDrive/alexandria_qwen35_sft/semantic_similarity_e5_large


In [4]:
# ============================================================
# Cell 3 — Main experiment configuration
# Qwen3-4B LoRA best checkpoint + ntkmirror controller
# ============================================================

from pathlib import Path
import json
import random
import numpy as np
import torch

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ------------------------------------------------------------
# Base model + LoRA adapter
# ------------------------------------------------------------

MODEL_NAME = "Qwen/Qwen3-4B-Base"
LOAD_IN_4BIT = True

# This should be the exact LoRA run folder that produced your best Qwen result.
LORA_EXPERIMENT_NAME = (
    "qwen3_4b_alexandria_eg_only_context3_"
    "complete2shot_all_group_r16_10epochs"
)

LORA_RUN_DIR = RUNS_DIR / LORA_EXPERIMENT_NAME

# We know your current best LoRA checkpoint is step 700.
MANUAL_LORA_BEST_STEP = 700

if MANUAL_LORA_BEST_STEP is not None:
    LORA_ADAPTER_PATH = LORA_RUN_DIR / f"checkpoint-{MANUAL_LORA_BEST_STEP}"
else:
    LORA_ADAPTER_PATH = None

# ------------------------------------------------------------
# Dataset setup
# ------------------------------------------------------------

DATASET_NAME = "UBC-NLP/alexandria"

SELECTED_CONFIGS_MODE = "EG_ONLY"  # "EG_ONLY", "ALL", or "MANUAL"
MANUAL_CONFIGS = ["EG"]

MAX_CONTEXT_TURNS = 3
USE_PREVIOUS_ENGLISH_CONTEXT = True
USE_METADATA = True

# ------------------------------------------------------------
# Prompting setup
# ------------------------------------------------------------

USE_FEW_SHOTS = True
N_FEW_SHOTS = 2
MAX_FEW_SHOT_EXAMPLE_CHARS = 450

MAX_SEQ_LENGTH = 2048
MAX_NEW_TOKENS = 120

# ------------------------------------------------------------
# ntkmirror controller setup
# ------------------------------------------------------------

NTK_GATES = 10000

# Serious run: all layers.
# If too slow, create a separate experiment with "last:18".
NTK_LAYERS = "all"

NTK_HOOK_SITE = "layer_output"
NTK_MAX_LOG_GATE = 0.05

NTK_SCORE_EXAMPLES = 256
NTK_L2 = 1e-5

# ------------------------------------------------------------
# Training hyperparameters
# ------------------------------------------------------------

NUM_EPOCHS = 5
PER_DEVICE_BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 8

LEARNING_RATE = 5e-4
WARMUP_RATIO = 0.03
WEIGHT_DECAY = 0.0

LOGGING_STEPS = 10
EVAL_STEPS = 100
SAVE_STEPS = 100
SAVE_TOTAL_LIMIT = 50

EVAL_NLL_LIMIT = 128

# ------------------------------------------------------------
# Metric checkpoint sweep
# ------------------------------------------------------------

SWEEP_EVAL_LIMIT = None
SWEEP_STEPS = [ 600, 1000, 1600]

PRIMARY_SELECTION_METRIC = "chrF++"
SECONDARY_SELECTION_METRIC = "spBLEU"

# ------------------------------------------------------------
# Experiment name
# ------------------------------------------------------------

LORA_STEP_FOR_NAME = (
    MANUAL_LORA_BEST_STEP
    if MANUAL_LORA_BEST_STEP is not None
    else "auto"
)

EXPERIMENT_NAME = (
    f"qwen3_4b_lora_step{LORA_STEP_FOR_NAME}_"
    f"alexandria_eg_only_context3_complete2shot_"
    f"ntkmirror_PREVSTYLE_"
    f"g{NTK_GATES}_score{NTK_SCORE_EXAMPLES}_"
    f"layers_{NTK_LAYERS.replace(':', '')}_"
    f"mlgate{str(NTK_MAX_LOG_GATE).replace('.', '')}_"
    f"lr{str(LEARNING_RATE).replace('.', 'p')}_"
    f"{NUM_EPOCHS}epochs"
)

OUTPUT_DIR = RUNS_DIR / EXPERIMENT_NAME
CONTROLLER_DIR = ADAPTER_DIR / EXPERIMENT_NAME

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CONTROLLER_DIR.mkdir(parents=True, exist_ok=True)

print("MODEL_NAME:", MODEL_NAME)
print("LORA_EXPERIMENT_NAME:", LORA_EXPERIMENT_NAME)
print("LORA_RUN_DIR:", LORA_RUN_DIR)
print("MANUAL_LORA_BEST_STEP:", MANUAL_LORA_BEST_STEP)
print("LORA_ADAPTER_PATH:", LORA_ADAPTER_PATH)

print("\nEXPERIMENT_NAME:", EXPERIMENT_NAME)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("CONTROLLER_DIR:", CONTROLLER_DIR)

print("\nNTK config:")
print("  NTK_LAYERS:", NTK_LAYERS)
print("  NTK_GATES:", NTK_GATES)
print("  NTK_MAX_LOG_GATE:", NTK_MAX_LOG_GATE)
print("  NTK_SCORE_EXAMPLES:", NTK_SCORE_EXAMPLES)

MODEL_NAME: Qwen/Qwen3-4B-Base
LORA_EXPERIMENT_NAME: qwen3_4b_alexandria_eg_only_context3_complete2shot_all_group_r16_10epochs
LORA_RUN_DIR: /content/drive/MyDrive/alexandria_qwen35_sft/runs/qwen3_4b_alexandria_eg_only_context3_complete2shot_all_group_r16_10epochs
MANUAL_LORA_BEST_STEP: 700
LORA_ADAPTER_PATH: /content/drive/MyDrive/alexandria_qwen35_sft/runs/qwen3_4b_alexandria_eg_only_context3_complete2shot_all_group_r16_10epochs/checkpoint-700

EXPERIMENT_NAME: qwen3_4b_lora_step700_alexandria_eg_only_context3_complete2shot_ntkmirror_PREVSTYLE_g10000_score256_layers_all_mlgate005_lr0p0005_5epochs
OUTPUT_DIR: /content/drive/MyDrive/alexandria_qwen35_sft/runs/qwen3_4b_lora_step700_alexandria_eg_only_context3_complete2shot_ntkmirror_PREVSTYLE_g10000_score256_layers_all_mlgate005_lr0p0005_5epochs
CONTROLLER_DIR: /content/drive/MyDrive/alexandria_qwen35_sft/final_adapters/qwen3_4b_lora_step700_alexandria_eg_only_context3_complete2shot_ntkmirror_PREVSTYLE_g10000_score256_layers_all_mlgate0

In [5]:
# ============================================================
# Cell 3B — Resolve best LoRA adapter checkpoint
# ============================================================

import json
from pathlib import Path

def checkpoint_step(path):
    path = Path(path)
    name = path.name

    if not name.startswith("checkpoint-"):
        return -1

    try:
        return int(name.replace("checkpoint-", ""))
    except Exception:
        return -1


def read_json_safe(path):
    path = Path(path)

    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except Exception as e:
        print(f"WARNING: could not read JSON file: {path}")
        print("Reason:", repr(e))
        return None


def find_trainer_state_files(run_dir, checkpoint_dir=None):
    run_dir = Path(run_dir)
    candidates = []

    if checkpoint_dir is not None:
        checkpoint_dir = Path(checkpoint_dir)
        candidates.extend([
            checkpoint_dir / "trainer_state.json",
            checkpoint_dir.parent / "trainer_state.json",
        ])

    candidates.extend([
        run_dir / "trainer_state.json",
        run_dir / "trainer_state_best.json",
    ])

    # Also search recursively as fallback.
    candidates.extend(list(run_dir.rglob("trainer_state.json")))

    # Deduplicate while preserving order.
    seen = set()
    unique = []

    for p in candidates:
        p = Path(p)
        key = str(p)

        if key in seen:
            continue

        seen.add(key)

        if p.exists():
            unique.append(p)

    return unique


def extract_eval_loss_for_step(state, step):
    if state is None:
        return None

    step = int(step)

    matched_losses = []

    for item in state.get("log_history", []):
        try:
            item_step = int(item.get("step", -1))
        except Exception:
            continue

        if item_step != step:
            continue

        if "eval_loss" in item:
            matched_losses.append(float(item["eval_loss"]))
        elif "eval_nll" in item:
            matched_losses.append(float(item["eval_nll"]))

    if matched_losses:
        return matched_losses[-1]

    return None


def find_best_lora_checkpoint(output_dir):
    output_dir = Path(output_dir)

    state_files = find_trainer_state_files(output_dir)

    if not state_files:
        raise FileNotFoundError(f"No trainer_state.json found under: {output_dir}")

    readable_states = []

    for sf in state_files:
        state = read_json_safe(sf)

        if state is None:
            continue

        global_step = int(state.get("global_step", -1))
        readable_states.append((global_step, sf, state))

    if not readable_states:
        raise RuntimeError("Could not read a valid trainer_state.json")

    # Prefer the trainer_state with the largest global_step.
    readable_states = sorted(readable_states, key=lambda x: x[0], reverse=True)

    _, best_state_file, best_state = readable_states[0]

    official_best = best_state.get("best_model_checkpoint", None)
    official_metric = best_state.get("best_metric", None)

    if official_best is not None:
        official_best_path = Path(official_best)

        if not official_best_path.exists():
            official_best_path = output_dir / official_best_path.name

        if official_best_path.exists():
            return (
                official_best_path,
                checkpoint_step(official_best_path),
                official_metric,
                best_state_file,
            )

    eval_rows = []

    for item in best_state.get("log_history", []):
        if "step" not in item:
            continue

        if "eval_loss" in item:
            eval_rows.append({
                "step": int(item["step"]),
                "eval_loss": float(item["eval_loss"]),
            })
        elif "eval_nll" in item:
            eval_rows.append({
                "step": int(item["step"]),
                "eval_loss": float(item["eval_nll"]),
            })

    if not eval_rows:
        raise RuntimeError("No eval_loss/eval_nll records found in LoRA trainer_state.json")

    best_row = min(eval_rows, key=lambda x: x["eval_loss"])
    best_step = int(best_row["step"])
    best_path = output_dir / f"checkpoint-{best_step}"

    if not best_path.exists():
        raise FileNotFoundError(f"Inferred best LoRA checkpoint missing: {best_path}")

    return best_path, best_step, best_row["eval_loss"], best_state_file


def find_lora_metadata_for_manual_checkpoint(run_dir, adapter_path, step):
    run_dir = Path(run_dir)
    adapter_path = Path(adapter_path)
    step = int(step)

    state_files = find_trainer_state_files(run_dir, checkpoint_dir=adapter_path)

    best_state_file = None
    best_eval_loss = None

    for sf in state_files:
        state = read_json_safe(sf)

        if state is None:
            continue

        # First: try exact eval loss for this step.
        eval_loss = extract_eval_loss_for_step(state, step)

        if eval_loss is not None:
            return eval_loss, sf

        # Second: if trainer_state says this is best_model_checkpoint, use best_metric.
        official_best = state.get("best_model_checkpoint", None)

        if official_best is not None:
            official_best_path = Path(official_best)

            official_step = checkpoint_step(official_best_path)

            if official_step == step and state.get("best_metric") is not None:
                return float(state["best_metric"]), sf

        # Keep first readable state file as metadata source even if eval not found.
        if best_state_file is None:
            best_state_file = sf

    return best_eval_loss, best_state_file


# ------------------------------------------------------------
# Resolve adapter path
# ------------------------------------------------------------

if LORA_ADAPTER_PATH is None:
    (
        LORA_ADAPTER_PATH,
        LORA_BEST_STEP,
        LORA_BEST_EVAL_LOSS,
        LORA_STATE_FILE,
    ) = find_best_lora_checkpoint(LORA_RUN_DIR)

else:
    LORA_ADAPTER_PATH = Path(LORA_ADAPTER_PATH)
    LORA_BEST_STEP = checkpoint_step(LORA_ADAPTER_PATH)

    if LORA_BEST_STEP < 0:
        if MANUAL_LORA_BEST_STEP is None:
            raise ValueError(
                f"Could not infer checkpoint step from LORA_ADAPTER_PATH={LORA_ADAPTER_PATH}. "
                "Set MANUAL_LORA_BEST_STEP explicitly."
            )

        LORA_BEST_STEP = int(MANUAL_LORA_BEST_STEP)

    (
        LORA_BEST_EVAL_LOSS,
        LORA_STATE_FILE,
    ) = find_lora_metadata_for_manual_checkpoint(
        run_dir=LORA_RUN_DIR,
        adapter_path=LORA_ADAPTER_PATH,
        step=LORA_BEST_STEP,
    )


# ------------------------------------------------------------
# Validate adapter path/files
# ------------------------------------------------------------

LORA_ADAPTER_PATH = Path(LORA_ADAPTER_PATH)

if not LORA_ADAPTER_PATH.exists():
    raise FileNotFoundError(
        f"LoRA adapter path does not exist:\n{LORA_ADAPTER_PATH}\n\n"
        f"Check LORA_EXPERIMENT_NAME and MANUAL_LORA_BEST_STEP in Cell 3."
    )

adapter_config = LORA_ADAPTER_PATH / "adapter_config.json"
adapter_safetensors = LORA_ADAPTER_PATH / "adapter_model.safetensors"
adapter_bin = LORA_ADAPTER_PATH / "adapter_model.bin"

if not adapter_config.exists():
    raise FileNotFoundError(f"Missing adapter_config.json:\n{adapter_config}")

if not adapter_safetensors.exists() and not adapter_bin.exists():
    raise FileNotFoundError(
        "Missing LoRA adapter weights. Expected one of:\n"
        f"  {adapter_safetensors}\n"
        f"  {adapter_bin}"
    )

print("LoRA adapter files look OK.")

print("Resolved LoRA adapter:")
print("  path:", LORA_ADAPTER_PATH)
print("  step:", LORA_BEST_STEP)

if LORA_BEST_EVAL_LOSS is None:
    print("  eval_loss: not found / not required")
else:
    print("  eval_loss:", LORA_BEST_EVAL_LOSS)

if LORA_STATE_FILE is None:
    print("  state_file: not found / adapter-only checkpoint is OK")
else:
    print("  state_file:", LORA_STATE_FILE)

LoRA adapter files look OK.
Resolved LoRA adapter:
  path: /content/drive/MyDrive/alexandria_qwen35_sft/runs/qwen3_4b_alexandria_eg_only_context3_complete2shot_all_group_r16_10epochs/checkpoint-700
  step: 700
  eval_loss: 1.6058480739593506
  state_file: /content/drive/MyDrive/alexandria_qwen35_sft/runs/qwen3_4b_alexandria_eg_only_context3_complete2shot_all_group_r16_10epochs/checkpoint-700/trainer_state.json


### **Dataset Preparations**

In [6]:
# ============================================================
# Cell 4 — Load Alexandria configs
# ============================================================

from datasets import load_dataset, get_dataset_config_names
import pandas as pd

DATASET_NAME = "UBC-NLP/alexandria"

available_configs = get_dataset_config_names(DATASET_NAME)
print("Available configs:")
print(available_configs)

if SELECTED_CONFIGS_MODE == "EG_ONLY":
    selected_configs = ["EG"] if "EG" in available_configs else [available_configs[0]]

elif SELECTED_CONFIGS_MODE == "ALL":
    selected_configs = available_configs

elif SELECTED_CONFIGS_MODE == "MANUAL":
    selected_configs = MANUAL_CONFIGS
    missing = [c for c in selected_configs if c not in available_configs]
    if missing:
        raise ValueError(f"Unavailable configs: {missing}")

else:
    raise ValueError("SELECTED_CONFIGS_MODE must be EG_ONLY, ALL, or MANUAL.")

print("\nSelected configs:")
print(selected_configs)

loaded = {}

for cfg in selected_configs:
    print(f"\nLoading config: {cfg}")

    ds_train = load_dataset(DATASET_NAME, name=cfg, split="train")
    ds_test = load_dataset(DATASET_NAME, name=cfg, split="test")

    loaded[cfg] = {
        "train": ds_train,
        "test": ds_test,
    }

    print("Train:", ds_train)
    print("Test:", ds_test)
    print("Keys:", ds_train[0].keys())

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/24.5k [00:00<?, ?B/s]

Available configs:
['EG', 'JO', 'LB', 'LY', 'MA', 'MR', 'OM', 'PS', 'SA', 'SD', 'SY', 'TN', 'YE']

Selected configs:
['EG']

Loading config: EG


EG/train-00000-of-00001.parquet:   0%|          | 0.00/496k [00:00<?, ?B/s]

EG/test-00000-of-00001.parquet:   0%|          | 0.00/196k [00:00<?, ?B/s]

EG/dev-00000-of-00001.parquet:   0%|          | 0.00/183k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/982 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/366 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/352 [00:00<?, ? examples/s]

Train: Dataset({
    features: ['conv_id', 'country', 'domain', 'dialect', 'participants', 'english_conversation', 'dialectal_conversation', 'translator_id', 'reviewer_id'],
    num_rows: 982
})
Test: Dataset({
    features: ['conv_id', 'country', 'domain', 'dialect', 'participants', 'english_conversation', 'dialectal_conversation', 'translator_id', 'reviewer_id'],
    num_rows: 366
})
Keys: dict_keys(['conv_id', 'country', 'domain', 'dialect', 'participants', 'english_conversation', 'dialectal_conversation', 'translator_id', 'reviewer_id'])


In [7]:
# ============================================================
# Cell 5 — Flatten Alexandria conversations
# ============================================================

def safe_get(row, keys, default=""):
    for k in keys:
        if isinstance(row, dict) and k in row and row[k] is not None:
            return row[k]
    return default

def turn_text(turn):
    if isinstance(turn, dict):
        for k in ["text", "sentence", "utterance", "content", "value"]:
            if k in turn and turn[k] is not None:
                return str(turn[k]).strip()
        return str(turn).strip()
    return str(turn).strip()

def turn_field(turn, keys, default=""):
    if isinstance(turn, dict):
        for k in keys:
            if k in turn and turn[k] is not None:
                return str(turn[k]).strip()
    return default

def normalize_list(x):
    if x is None:
        return []
    if isinstance(x, list):
        return x
    return list(x)

def flatten_alexandria_split(ds, cfg_name, split_name, max_context_turns=3):
    records = []

    for conv_idx, row in enumerate(ds):
        english_conv = normalize_list(row["english_conversation"])
        dialect_conv = normalize_list(row["dialectal_conversation"])

        n = min(len(english_conv), len(dialect_conv))

        country = safe_get(row, ["country", "country_code"], cfg_name)
        dialect = safe_get(row, ["dialect", "dialect_label", "subdialect", "city", "variety"], "")
        domain = safe_get(row, ["domain", "topic"], "")
        persona = safe_get(row, ["persona", "roles", "speaker_roles"], "")
        conv_id = safe_get(row, ["conversation_id", "id", "dialogue_id"], f"{cfg_name}_{split_name}_{conv_idx}")

        for i in range(n):
            en_turn = english_conv[i]
            ar_turn = dialect_conv[i]

            source_text = turn_text(en_turn)
            target_text = turn_text(ar_turn)

            if not source_text or not target_text:
                continue

            prev_start = max(0, i - max_context_turns)
            prev_en_turns = english_conv[prev_start:i]

            previous_context = []
            for j, t in enumerate(prev_en_turns, start=prev_start):
                previous_context.append({
                    "turn_id": j,
                    "speaker": turn_field(t, ["speaker", "role", "speaker_role"], ""),
                    "direction": turn_field(t, ["direction", "gender_direction", "speaker_addressee_gender"], ""),
                    "text": turn_text(t),
                })

            records.append({
                "source_id": f"{cfg_name}_{split_name}_{conv_id}_{i}",
                "config": cfg_name,
                "split": split_name,
                "conversation_id": conv_id,
                "turn_id": i,

                "country": country,
                "dialect": dialect,
                "domain": domain,
                "persona": persona,

                "speaker": turn_field(en_turn, ["speaker", "role", "speaker_role"], ""),
                "gender_direction": turn_field(en_turn, ["direction", "gender_direction", "speaker_addressee_gender"], ""),

                "previous_english_turns": previous_context,
                "source_text": source_text,
                "target_arabic": target_text,
            })

    return records

train_records = []
eval_records = []

for cfg in selected_configs:
    train_records.extend(
        flatten_alexandria_split(
            loaded[cfg]["train"],
            cfg_name=cfg,
            split_name="train",
            max_context_turns=MAX_CONTEXT_TURNS,
        )
    )

    eval_records.extend(
        flatten_alexandria_split(
            loaded[cfg]["test"],
            cfg_name=cfg,
            split_name="test",
            max_context_turns=MAX_CONTEXT_TURNS,
        )
    )

train_df = pd.DataFrame(train_records)
eval_df = pd.DataFrame(eval_records)

print("Train shape:", train_df.shape)
print("Eval shape:", eval_df.shape)

print("\nTrain configs:")
print(train_df["config"].value_counts())

print("\nEval configs:")
print(eval_df["config"].value_counts())

train_jsonl = DATA_DIR / f"alexandria_train_{SELECTED_CONFIGS_MODE.lower()}_context{MAX_CONTEXT_TURNS}.jsonl"
eval_jsonl = DATA_DIR / f"alexandria_eval_{SELECTED_CONFIGS_MODE.lower()}_context{MAX_CONTEXT_TURNS}.jsonl"

train_df.to_json(train_jsonl, orient="records", lines=True, force_ascii=False)
eval_df.to_json(eval_jsonl, orient="records", lines=True, force_ascii=False)

print("\nSaved:")
print(train_jsonl)
print(eval_jsonl)

display(train_df.head())

Train shape: (3108, 14)
Eval shape: (1118, 14)

Train configs:
config
EG    3108
Name: count, dtype: int64

Eval configs:
config
EG    1118
Name: count, dtype: int64

Saved:
/content/drive/MyDrive/alexandria_qwen35_sft/prepared_data/alexandria_train_eg_only_context3.jsonl
/content/drive/MyDrive/alexandria_qwen35_sft/prepared_data/alexandria_eval_eg_only_context3.jsonl


,source_id,config,split,conversation_id,turn_id,country,dialect,domain,persona,speaker,gender_direction,previous_english_turns,source_text,target_arabic
0,EG_train_EG_train_0_0,EG,train,EG_train_0,0,EG,Egyptian Arabic (Cairene) Dialect,Agriculture and farming,,Wholesale Buyer,male -> female,[],Good morning. I'm looking to source 10 tons of...,صباح الخير، عايز عشرة طن من الخرشوف الكويس للت...
1,EG_train_EG_train_0_1,EG,train,EG_train_0,1,EG,Egyptian Arabic (Cairene) Dialect,Agriculture and farming,,Wholesale Seller,female -> male,"[{'turn_id': 0, 'speaker': 'Wholesale Buyer', ...",Good morning to you. You heard correctly. My a...,صباح النور،سمعك مظبوط،الخرشوف بتاعي من أحسن ال...
2,EG_train_EG_train_0_2,EG,train,EG_train_0,2,EG,Egyptian Arabic (Cairene) Dialect,Agriculture and farming,,Wholesale Buyer,male -> female,"[{'turn_id': 0, 'speaker': 'Wholesale Buyer', ...","Excellent. Yes, please show me. I need them to...",ممتاز، لو سمحتي وريني، عايزه بمقاس واحد ومافيه...
3,EG_train_EG_train_0_3,EG,train,EG_train_0,3,EG,Egyptian Arabic (Cairene) Dialect,Agriculture and farming,,Wholesale Seller,female -> male,"[{'turn_id': 0, 'speaker': 'Wholesale Buyer', ...",Don't you worry. You will be very satisfied. M...,متخافش، هتنبسط جدا، سمعتي جاية من الحاجة الكويسة.
4,EG_train_EG_train_1_0,EG,train,EG_train_1,0,EG,Egyptian Arabic (Cairene) Dialect,Agriculture and farming,,Farmer,female -> female,[],I usually use the regular granular fertilizer....,أنا عادة بستخدم السماد العادي الحبيبات. ايه فا...


In [8]:
# ============================================================
# Cell 6 — Build prompts/messages with complete 2-shot examples
# Must match the Qwen3-4B LoRA run style
# ============================================================

from datasets import Dataset
import hashlib
import pandas as pd

SYSTEM_PROMPT = (
    "You are a professional machine translation system. "
    "Translate the current English dialogue turn into natural dialectal Arabic. "
    "Use the provided training examples only as style and dialect guidance. "
    "Return only the translation, without explanation."
)

def build_context(previous_turns):
    if not USE_PREVIOUS_ENGLISH_CONTEXT or not previous_turns:
        return "No previous context."

    lines = []
    for i, t in enumerate(previous_turns, start=1):
        speaker = t.get("speaker", "")
        text = t.get("text", "")

        if speaker:
            lines.append(f"{i}. {speaker}: {text}")
        else:
            lines.append(f"{i}. {text}")

    return "\n".join(lines)

def build_metadata_block(row):
    if not USE_METADATA:
        return "No metadata."

    fields = [
        ("Country/config", row.get("config", "")),
        ("Target dialect", row.get("dialect", "")),
        ("Domain", row.get("domain", "")),
        ("Persona/Roles", row.get("persona", "")),
        ("Current speaker", row.get("speaker", "")),
        ("Speaker-to-addressee gender direction", row.get("gender_direction", "")),
    ]

    lines = []
    for k, v in fields:
        v = str(v).strip()
        if v:
            lines.append(f"{k}: {v}")

    return "\n".join(lines) if lines else "No metadata."

def deterministic_seed_from_id(source_id, base_seed=SEED):
    raw = f"{source_id}_{base_seed}".encode("utf-8")
    return int(hashlib.md5(raw).hexdigest()[:8], 16)

def select_two_shots_from_train(row, train_pool, n=N_FEW_SHOTS):
    if not USE_FEW_SHOTS or n <= 0:
        return []

    row_source_id = str(row.get("source_id", ""))
    row_config = str(row.get("config", ""))
    row_domain = str(row.get("domain", ""))

    pool = train_pool.copy()
    pool["source_id"] = pool["source_id"].astype(str)
    pool = pool[pool["source_id"] != row_source_id].copy()

    if len(pool) == 0:
        return []

    pool["fewshot_total_chars"] = (
        pool["source_text"].astype(str).str.len()
        + pool["target_arabic"].astype(str).str.len()
    )

    short_pool = pool[pool["fewshot_total_chars"] <= MAX_FEW_SHOT_EXAMPLE_CHARS].copy()

    same_config_domain_short = short_pool[
        (short_pool["config"].astype(str) == row_config)
        & (short_pool["domain"].astype(str) == row_domain)
    ]

    same_config_short = short_pool[
        short_pool["config"].astype(str) == row_config
    ]

    same_config_domain = pool[
        (pool["config"].astype(str) == row_config)
        & (pool["domain"].astype(str) == row_domain)
    ]

    same_config = pool[
        pool["config"].astype(str) == row_config
    ]

    candidate_pools = [
        same_config_domain_short,
        same_config_short,
        short_pool,
        same_config_domain,
        same_config,
        pool,
    ]

    candidates = None
    for candidate_pool in candidate_pools:
        if len(candidate_pool) >= n:
            candidates = candidate_pool
            break

    if candidates is None:
        candidates = pool

    sample_n = min(n, len(candidates))
    seed = deterministic_seed_from_id(row_source_id)

    shots = candidates.sample(n=sample_n, random_state=seed)

    keep_cols = [
        "source_id",
        "config",
        "dialect",
        "domain",
        "source_text",
        "target_arabic",
    ]

    return shots[keep_cols].to_dict("records")

def build_few_shot_block(few_shot_examples):
    if not USE_FEW_SHOTS or not few_shot_examples:
        return "No examples available."

    blocks = []

    for i, ex in enumerate(few_shot_examples, start=1):
        ex_config = str(ex.get("config", "")).strip()
        ex_dialect = str(ex.get("dialect", "")).strip()
        ex_domain = str(ex.get("domain", "")).strip()

        meta_parts = []
        if ex_config:
            meta_parts.append(f"config={ex_config}")
        if ex_dialect:
            meta_parts.append(f"dialect={ex_dialect}")
        if ex_domain:
            meta_parts.append(f"domain={ex_domain}")

        meta_line = ", ".join(meta_parts) if meta_parts else "no metadata"

        ex_source = str(ex.get("source_text", "")).strip()
        ex_target = str(ex.get("target_arabic", "")).strip()

        blocks.append(
            f"""Example {i} ({meta_line})
English:
{ex_source}

Arabic:
{ex_target}"""
        )

    return "\n\n".join(blocks)

def make_user_prompt(row):
    context = build_context(row["previous_english_turns"])
    metadata = build_metadata_block(row)
    few_shots = build_few_shot_block(row.get("few_shot_examples", []))

    return f"""Task:
Translate the current English dialogue turn into the target dialectal Arabic variety.

Few-shot training examples:
{few_shots}

Metadata:
{metadata}

Previous English dialogue context:
{context}

Current English turn:
{row["source_text"]}

Rules:
- Preserve the meaning exactly.
- Use the target local dialect, not Modern Standard Arabic unless it is natural in context.
- Follow the dialect/style pattern shown in the few-shot examples when relevant.
- Do not copy the few-shot examples.
- Preserve names, numbers, named entities, and technical terms when appropriate.
- Keep the tone appropriate for the speaker and domain.
- Return only the Arabic translation."""

def row_to_messages(row):
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": make_user_prompt(row)},
        {"role": "assistant", "content": row["target_arabic"]},
    ]

few_shot_pool_cols = [
    "source_id",
    "config",
    "dialect",
    "domain",
    "source_text",
    "target_arabic",
]

train_few_shot_pool = train_df[few_shot_pool_cols].copy()

train_df["few_shot_examples"] = train_df.apply(
    lambda row: select_two_shots_from_train(row, train_few_shot_pool, n=N_FEW_SHOTS),
    axis=1,
)

eval_df["few_shot_examples"] = eval_df.apply(
    lambda row: select_two_shots_from_train(row, train_few_shot_pool, n=N_FEW_SHOTS),
    axis=1,
)

train_df["messages"] = train_df.apply(row_to_messages, axis=1)
eval_df["messages"] = eval_df.apply(row_to_messages, axis=1)

train_dataset = Dataset.from_pandas(
    train_df[
        [
            "messages",
            "source_id",
            "config",
            "domain",
            "dialect",
            "source_text",
            "target_arabic",
        ]
    ],
    preserve_index=False,
)

eval_dataset = Dataset.from_pandas(
    eval_df[
        [
            "messages",
            "source_id",
            "config",
            "domain",
            "dialect",
            "source_text",
            "target_arabic",
        ]
    ],
    preserve_index=False,
)

print(train_dataset)
print(eval_dataset)

print("\nExample few-shot source IDs:")
print([x["source_id"] for x in train_df.iloc[0]["few_shot_examples"]])

print("\nExample messages:")
train_dataset[0]["messages"]

Dataset({
    features: ['messages', 'source_id', 'config', 'domain', 'dialect', 'source_text', 'target_arabic'],
    num_rows: 3108
})
Dataset({
    features: ['messages', 'source_id', 'config', 'domain', 'dialect', 'source_text', 'target_arabic'],
    num_rows: 1118
})

Example few-shot source IDs:
['EG_train_EG_train_79_2', 'EG_train_EG_train_77_2']

Example messages:


[{'content': 'You are a professional machine translation system. Translate the current English dialogue turn into natural dialectal Arabic. Use the provided training examples only as style and dialect guidance. Return only the translation, without explanation.',
  'role': 'system'},
 {'content': "Task:\nTranslate the current English dialogue turn into the target dialectal Arabic variety.\n\nFew-shot training examples:\nExample 1 (config=EG, dialect=Egyptian Arabic (Cairene) Dialect, domain=Agriculture and farming)\nEnglish:\nYes, I believe the original inspection that led to my fine was not done correctly.\n\nArabic:\nأيوه، أنا شايف المعاينة الأولى اللي اتعملت واللي خدت بسببها الغرامة ما كانتش مظبوطة.\n\nExample 2 (config=EG, dialect=Egyptian Arabic (Cairene) Dialect, domain=Agriculture and farming)\nEnglish:\nYes, the project is heavily subsidized. Let me walk you through the proposal and the benefits.\n\nArabic:\nأيوه، المشروع مدعوم بنسبة كبيرة. خليني أشرحلك المقترح وفوايده.\n\nMetad

### Manual SFT/NTK format

In [9]:
# ============================================================
# Cell 7 — Manual SFT format used by the LoRA run
# ============================================================

SYSTEM_MARKER = "### System:"
INSTRUCTION_MARKER = "### Instruction:"
RESPONSE_MARKER = "### Arabic translation:"

def get_message_content(messages, role):
    for m in messages:
        if m.get("role") == role:
            return m.get("content", "")
    return ""

def format_ntkmirror_prompt(system_text, user_text):
    return (
        f"{SYSTEM_MARKER}\n"
        f"{system_text.strip()}\n\n"
        f"{INSTRUCTION_MARKER}\n"
        f"{user_text.strip()}\n\n"
        f"{RESPONSE_MARKER}\n"
    )

def format_sft_text(system_text, user_text, assistant_text=None, add_eos=True):
    text = format_ntkmirror_prompt(system_text, user_text)

    if assistant_text is not None:
        text += str(assistant_text).strip()

        if add_eos and tokenizer.eos_token is not None:
            text += tokenizer.eos_token

    return text

print("Manual template ready.")

Manual template ready.


### **Load Qwen base + LoRA adapter**

In [10]:
# ============================================================
# Cell 8 — Load Qwen3-4B Base + frozen LoRA adapter
# ============================================================

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

if LOAD_IN_4BIT:
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
    )
else:
    quantization_config = None

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    device_map="auto",
    trust_remote_code=True,
)

print("Base model loaded.")

model = PeftModel.from_pretrained(
    base_model,
    str(LORA_ADAPTER_PATH),
    is_trainable=False,
)

model.eval()
model.config.use_cache = False

for p in model.parameters():
    p.requires_grad = False

device = next(model.parameters()).device

print("Loaded model:")
print("  base:", MODEL_NAME)
print("  LoRA adapter:", LORA_ADAPTER_PATH)
print("  LoRA best step:", LORA_BEST_STEP)
print("  device:", device)
print("  pad token:", tokenizer.pad_token)
print("  eos token:", tokenizer.eos_token)

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.68k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

Base model loaded.
Loaded model:
  base: Qwen/Qwen3-4B-Base
  LoRA adapter: /content/drive/MyDrive/alexandria_qwen35_sft/runs/qwen3_4b_alexandria_eg_only_context3_complete2shot_all_group_r16_10epochs/checkpoint-700
  LoRA best step: 700
  device: cuda:0
  pad token: <|endoftext|>
  eos token: <|endoftext|>


In [11]:
# ============================================================
# Cell 9 — Sanity generation with LoRA only, before NTK
# ============================================================

import torch

def extract_answer(decoded_text):
    if RESPONSE_MARKER in decoded_text:
        answer = decoded_text.split(RESPONSE_MARKER)[-1]
    else:
        answer = decoded_text

    special_tokens = [
        tokenizer.eos_token,
        tokenizer.pad_token,
        "<|endoftext|>",
        "<|im_end|>",
    ]

    for tok in special_tokens:
        if tok:
            answer = answer.replace(tok, "")

    return answer.strip()

def generate_lora_only_from_row(row, max_new_tokens=MAX_NEW_TOKENS):
    user_text = make_user_prompt(row)

    prompt = format_ntkmirror_prompt(
        system_text=SYSTEM_PROMPT,
        user_text=user_text,
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.05,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=False)
    answer = extract_answer(decoded)

    return answer, decoded

sample = eval_df.sample(1, random_state=SEED).iloc[0].to_dict()

pred, raw = generate_lora_only_from_row(sample)

print("Config:", sample["config"])
print("Dialect:", sample["dialect"])
print("Domain:", sample["domain"])

print("\nEnglish:")
print(sample["source_text"])

print("\nReference Arabic:")
print(sample["target_arabic"])

print("\nLoRA-only prediction:")
print(pred)

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Config: EG
Dialect: Egyptian Arabic (Cairene) Dialect
Domain: Legal and financial

English:
And can I ask for financial compensation for the damage to my name?

Reference Arabic:
وأقدر أطالب بتعويض مادي عشان الضرر اللي حصل لسمعتي؟

LoRA-only prediction:
وكمان ممكن أطلب تعويض مادي عن الضرر اللي حصل علي اسميا؟


### **Build ntkmirror Examples**

In [12]:
# ============================================================
# Cell 10 — Build ntkmirror Example objects
# ============================================================

from ntkmirror import Example

def row_to_ntkmirror_example(row):
    messages = row["messages"]

    system_text = get_message_content(messages, "system")
    user_text = get_message_content(messages, "user")
    assistant_text = get_message_content(messages, "assistant")

    prompt = format_ntkmirror_prompt(system_text, user_text)
    completion = str(assistant_text).strip()

    if tokenizer.eos_token is not None:
        completion += tokenizer.eos_token

    return Example(prompt=prompt, completion=completion)

train_ntk_examples = [
    row_to_ntkmirror_example(train_dataset[i])
    for i in range(len(train_dataset))
]

eval_ntk_examples = [
    row_to_ntkmirror_example(eval_dataset[i])
    for i in range(len(eval_dataset))
]

print("train_ntk_examples:", len(train_ntk_examples))
print("eval_ntk_examples:", len(eval_ntk_examples))

print("\nExample prompt:")
print(train_ntk_examples[0].prompt[:1500])

print("\nExample completion:")
print(train_ntk_examples[0].completion[:500])

train_ntk_examples: 3108
eval_ntk_examples: 1118

Example prompt:
### System:
You are a professional machine translation system. Translate the current English dialogue turn into natural dialectal Arabic. Use the provided training examples only as style and dialect guidance. Return only the translation, without explanation.

### Instruction:
Task:
Translate the current English dialogue turn into the target dialectal Arabic variety.

Few-shot training examples:
Example 1 (config=EG, dialect=Egyptian Arabic (Cairene) Dialect, domain=Agriculture and farming)
English:
Yes, I believe the original inspection that led to my fine was not done correctly.

Arabic:
أيوه، أنا شايف المعاينة الأولى اللي اتعملت واللي خدت بسببها الغرامة ما كانتش مظبوطة.

Example 2 (config=EG, dialect=Egyptian Arabic (Cairene) Dialect, domain=Agriculture and farming)
English:
Yes, the project is heavily subsidized. Let me walk you through the proposal and the benefits.

Arabic:
أيوه، المشروع مدعوم بنسبة كبيرة. خليني أشر

### nitialize ntkmirror tuner

In [13]:
# ============================================================
# Cell 11 — Initialize ntkmirror ForwardFineTuner
# ============================================================

from ntkmirror import ForwardFineTuner

tuner = ForwardFineTuner(
    model=model,
    tokenizer=tokenizer,
    gates=NTK_GATES,
    layers=NTK_LAYERS,
    max_log_gate=NTK_MAX_LOG_GATE,
    hook_site=NTK_HOOK_SITE,
)

print("ForwardFineTuner ready.")
print("Layer path:", tuner.layer_path)
print("Number of decoder layers:", len(tuner.decoder_layers))
print("Hidden size:", tuner.hidden_size)
print("Selected layer ids:", tuner.layer_ids[:10], "..." if len(tuner.layer_ids) > 10 else "")

ForwardFineTuner ready.
Layer path: base_model.model.model.layers
Number of decoder layers: 36
Hidden size: 2560
Selected layer ids: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9] ...


### Check existing ntkmirror checkpoints

In [14]:
# ============================================================
# Cell 12 — Check for existing ntkmirror checkpoints
# Same style as previous NTK notebooks
# ============================================================

from pathlib import Path
import re

def checkpoint_step(path):
    m = re.search(r"checkpoint-(\d+)", str(path))
    return int(m.group(1)) if m else -1

def list_ntk_checkpoints(output_dir):
    output_dir = Path(output_dir)

    checkpoints = sorted(
        [
            p for p in output_dir.glob("checkpoint-*")
            if p.is_dir() and (p / "controller.pt").exists()
        ],
        key=checkpoint_step,
    )

    return [(checkpoint_step(p), p) for p in checkpoints]

def get_last_ntk_checkpoint(output_dir):
    checkpoints = list_ntk_checkpoints(output_dir)
    return checkpoints[-1][1] if checkpoints else None

last_checkpoint = get_last_ntk_checkpoint(OUTPUT_DIR)

if last_checkpoint:
    print("Found ntkmirror checkpoint:")
    print(last_checkpoint)
else:
    print("No ntkmirror checkpoint found. Training will start from scratch.")

print("\nExisting checkpoints:")
for step, p in list_ntk_checkpoints(OUTPUT_DIR):
    print(f"  step={step}: {p}")

Found ntkmirror checkpoint:
/content/drive/MyDrive/alexandria_qwen35_sft/runs/qwen3_4b_lora_step700_alexandria_eg_only_context3_complete2shot_ntkmirror_PREVSTYLE_g10000_score256_layers_all_mlgate005_lr0p0005_5epochs/checkpoint-1700

Existing checkpoints:
  step=100: /content/drive/MyDrive/alexandria_qwen35_sft/runs/qwen3_4b_lora_step700_alexandria_eg_only_context3_complete2shot_ntkmirror_PREVSTYLE_g10000_score256_layers_all_mlgate005_lr0p0005_5epochs/checkpoint-100
  step=200: /content/drive/MyDrive/alexandria_qwen35_sft/runs/qwen3_4b_lora_step700_alexandria_eg_only_context3_complete2shot_ntkmirror_PREVSTYLE_g10000_score256_layers_all_mlgate005_lr0p0005_5epochs/checkpoint-200
  step=300: /content/drive/MyDrive/alexandria_qwen35_sft/runs/qwen3_4b_lora_step700_alexandria_eg_only_context3_complete2shot_ntkmirror_PREVSTYLE_g10000_score256_layers_all_mlgate005_lr0p0005_5epochs/checkpoint-300
  step=400: /content/drive/MyDrive/alexandria_qwen35_sft/runs/qwen3_4b_lora_step700_alexandria_eg_on

### **Optimizer, scheduler, checkpoint helpers**

In [15]:
# ============================================================
# Cell 13 — Optimizer/scheduler/checkpoint helpers
# Same style as previous NTK notebooks
# ============================================================

import math
import json
import shutil
from pathlib import Path

import torch
from transformers import get_cosine_schedule_with_warmup

from ntkmirror.data import batches, make_batch
from ntkmirror.losses import causal_loss_from_logits, token_accuracy_from_logits

device = next(model.parameters()).device

num_micro_batches_per_epoch = math.ceil(len(train_ntk_examples) / PER_DEVICE_BATCH_SIZE)

TOTAL_OPTIMIZER_STEPS = math.ceil(
    num_micro_batches_per_epoch * NUM_EPOCHS / GRAD_ACCUM_STEPS
)

WARMUP_STEPS = int(TOTAL_OPTIMIZER_STEPS * WARMUP_RATIO)

print("Micro-batches per epoch:", num_micro_batches_per_epoch)
print("Total optimizer steps:", TOTAL_OPTIMIZER_STEPS)
print("Warmup steps:", WARMUP_STEPS)
print("Device:", device)

def evaluate_ntk_nll(examples, batch_size=1, max_length=MAX_SEQ_LENGTH, limit=None):
    if limit is not None:
        examples = examples[: int(limit)]

    return tuner.evaluate_nll(
        examples,
        batch_size=batch_size,
        max_length=max_length,
        use_controller=True,
    )

def save_ntk_checkpoint(
    checkpoint_dir,
    global_step,
    epoch,
    optimizer,
    scheduler,
    log_history,
    best_metric,
    best_model_checkpoint,
):
    checkpoint_dir = Path(checkpoint_dir)
    checkpoint_dir.mkdir(parents=True, exist_ok=True)

    controller_path = checkpoint_dir / "controller.pt"
    optimizer_path = checkpoint_dir / "optimizer.pt"
    scheduler_path = checkpoint_dir / "scheduler.pt"
    state_path = checkpoint_dir / "trainer_state.json"

    tuner.save(controller_path)

    torch.save(
        {
            "optimizer": optimizer.state_dict(),
            "global_step": int(global_step),
            "epoch": float(epoch),
        },
        optimizer_path,
    )

    torch.save(
        {
            "scheduler": scheduler.state_dict(),
            "global_step": int(global_step),
            "epoch": float(epoch),
        },
        scheduler_path,
    )

    state = {
        "global_step": int(global_step),
        "epoch": float(epoch),
        "best_metric": None if best_metric is None else float(best_metric),
        "best_model_checkpoint": None if best_model_checkpoint is None else str(best_model_checkpoint),
        "log_history": log_history,
        "total_optimizer_steps": int(TOTAL_OPTIMIZER_STEPS),
        "num_train_epochs": int(NUM_EPOCHS),
        "learning_rate": float(LEARNING_RATE),
        "per_device_train_batch_size": int(PER_DEVICE_BATCH_SIZE),
        "gradient_accumulation_steps": int(GRAD_ACCUM_STEPS),
        "ntk_gates": int(NTK_GATES),
        "ntk_layers": str(NTK_LAYERS),
        "ntk_max_log_gate": float(NTK_MAX_LOG_GATE),
        "ntk_hook_site": str(NTK_HOOK_SITE),
        "lora_adapter_path": str(LORA_ADAPTER_PATH),
        "lora_best_step": int(LORA_BEST_STEP),
        "experiment_name": EXPERIMENT_NAME,
    }

    state_path.write_text(
        json.dumps(state, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )

def load_ntk_checkpoint(checkpoint_dir):
    checkpoint_dir = Path(checkpoint_dir)

    controller_path = checkpoint_dir / "controller.pt"
    optimizer_path = checkpoint_dir / "optimizer.pt"
    scheduler_path = checkpoint_dir / "scheduler.pt"
    state_path = checkpoint_dir / "trainer_state.json"

    if not controller_path.exists():
        raise FileNotFoundError(controller_path)

    tuner.load(controller_path, allow_model_mismatch=True)

    state = json.loads(state_path.read_text(encoding="utf-8")) if state_path.exists() else {}

    opt_state = torch.load(optimizer_path, map_location="cpu") if optimizer_path.exists() else None
    sch_state = torch.load(scheduler_path, map_location="cpu") if scheduler_path.exists() else None

    return state, opt_state, sch_state

def prune_old_checkpoints(output_dir, save_total_limit, best_model_checkpoint=None):
    output_dir = Path(output_dir)

    checkpoints = sorted(
        [
            p for p in output_dir.glob("checkpoint-*")
            if p.is_dir() and (p / "controller.pt").exists()
        ],
        key=checkpoint_step,
    )

    if save_total_limit is None or save_total_limit <= 0:
        return

    best_model_checkpoint = str(best_model_checkpoint) if best_model_checkpoint else None

    while len(checkpoints) > save_total_limit:
        candidate = checkpoints.pop(0)

        if best_model_checkpoint and str(candidate) == best_model_checkpoint:
            checkpoints.append(candidate)
            checkpoints = sorted(checkpoints, key=checkpoint_step)

            non_best = [p for p in checkpoints if str(p) != best_model_checkpoint]

            if not non_best:
                break

            candidate = non_best[0]
            checkpoints.remove(candidate)

        print("Pruning old checkpoint:", candidate)
        shutil.rmtree(candidate, ignore_errors=True)

print("Checkpoint helpers ready.")

Micro-batches per epoch: 3108
Total optimizer steps: 1943
Warmup steps: 58
Device: cuda:0
Checkpoint helpers ready.


### **Train / resume NTK controller**

In [16]:
# ============================================================
# Cell 14 — Initialize or resume ntkmirror controller
# Same style as previous NTK notebooks
# ============================================================

import math
import random
import torch

last_checkpoint = get_last_ntk_checkpoint(OUTPUT_DIR)

if last_checkpoint:
    print("Resuming from:", last_checkpoint)

    saved_state, opt_state, sch_state = load_ntk_checkpoint(last_checkpoint)

    START_GLOBAL_STEP = int(saved_state.get("global_step", checkpoint_step(last_checkpoint)))
    START_EPOCH_FLOAT = float(saved_state.get("epoch", 0.0))

    log_history = list(saved_state.get("log_history", []))

    BEST_EVAL_LOSS_SO_FAR = saved_state.get("best_metric", None)
    BEST_CHECKPOINT_SO_FAR = saved_state.get("best_model_checkpoint", None)

    print("Loaded controller from checkpoint.")
    print("Start global step:", START_GLOBAL_STEP)
    print("Best eval loss so far:", BEST_EVAL_LOSS_SO_FAR)
    print("Best checkpoint so far:", BEST_CHECKPOINT_SO_FAR)

else:
    print("No checkpoint found for this Qwen+LoRA NTK experiment.")
    print("Starting fresh controller initialization.")

    rng = random.Random(SEED)

    score_k = min(int(NTK_SCORE_EXAMPLES), len(train_ntk_examples))

    scoring_indices = rng.sample(
        range(len(train_ntk_examples)),
        k=score_k,
    )

    score_ntk_examples = [train_ntk_examples[i] for i in scoring_indices]

    effective_score_batches = math.ceil(
        len(score_ntk_examples) / PER_DEVICE_BATCH_SIZE
    )

    print("\nGate scoring setup:")
    print("  total train examples:", len(train_ntk_examples))
    print("  scoring examples:", len(score_ntk_examples))
    print("  per-device batch size:", PER_DEVICE_BATCH_SIZE)
    print("  effective score batches:", effective_score_batches)
    print("  max length:", MAX_SEQ_LENGTH)
    print("  gates:", NTK_GATES)
    print("  layers:", NTK_LAYERS)
    print("  device:", device)

    print("\nInitializing ntkmirror controller by activation-gradient gate scoring...")
    print("This can be silent for several minutes.")

    init_stats = tuner.initialize_controller(
        score_ntk_examples,
        score_batches=effective_score_batches,
        batch_size=PER_DEVICE_BATCH_SIZE,
        max_length=MAX_SEQ_LENGTH,
    )

    print("\nController initialization stats:")
    print(init_stats)

    START_GLOBAL_STEP = 0
    START_EPOCH_FLOAT = 0.0
    log_history = []
    BEST_EVAL_LOSS_SO_FAR = None
    BEST_CHECKPOINT_SO_FAR = None
    opt_state = None
    sch_state = None

if tuner.controller is None:
    raise RuntimeError("ntkmirror controller was not initialized or loaded.")

controller_gate_count = int(tuner.controller.raw.numel())

print("\nController check:")
print("  expected gates:", NTK_GATES)
print("  actual gates:", controller_gate_count)
print("  max_log_gate:", getattr(tuner.controller, "max_log_gate", NTK_MAX_LOG_GATE))
print("  hook_site:", getattr(tuner.controller, "hook_site", NTK_HOOK_SITE))

if controller_gate_count != int(NTK_GATES):
    raise RuntimeError(
        f"Loaded controller has {controller_gate_count} gates, "
        f"but current config expects {NTK_GATES}. "
        "This usually means you are accidentally resuming an incompatible experiment."
    )

# ------------------------------------------------------------
# Optimizer and scheduler
# ------------------------------------------------------------

optimizer = torch.optim.AdamW(
    [tuner.controller.raw],
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=WARMUP_STEPS,
    num_training_steps=TOTAL_OPTIMIZER_STEPS,
)

if opt_state is not None:
    optimizer.load_state_dict(opt_state["optimizer"])
    print("\nLoaded optimizer state.")

if sch_state is not None:
    scheduler.load_state_dict(sch_state["scheduler"])
    print("Loaded scheduler state.")

print("\nTraining state:")
print("  start global step:", START_GLOBAL_STEP)
print("  total optimizer steps:", TOTAL_OPTIMIZER_STEPS)
print("  warmup steps:", WARMUP_STEPS)
print("  learning rate:", LEARNING_RATE)
print("  best eval loss so far:", BEST_EVAL_LOSS_SO_FAR)
print("  best checkpoint so far:", BEST_CHECKPOINT_SO_FAR)

Resuming from: /content/drive/MyDrive/alexandria_qwen35_sft/runs/qwen3_4b_lora_step700_alexandria_eg_only_context3_complete2shot_ntkmirror_PREVSTYLE_g10000_score256_layers_all_mlgate005_lr0p0005_5epochs/checkpoint-1700
Loaded controller from checkpoint.
Start global step: 1700
Best eval loss so far: 1.6108958493079024
Best checkpoint so far: /content/drive/MyDrive/alexandria_qwen35_sft/runs/qwen3_4b_lora_step700_alexandria_eg_only_context3_complete2shot_ntkmirror_PREVSTYLE_g10000_score256_layers_all_mlgate005_lr0p0005_5epochs/checkpoint-100

Controller check:
  expected gates: 10000
  actual gates: 10000
  max_log_gate: 0.05
  hook_site: layer_output

Loaded optimizer state.
Loaded scheduler state.

Training state:
  start global step: 1700
  total optimizer steps: 1943
  warmup steps: 58
  learning rate: 0.0005
  best eval loss so far: 1.6108958493079024
  best checkpoint so far: /content/drive/MyDrive/alexandria_qwen35_sft/runs/qwen3_4b_lora_step700_alexandria_eg_only_context3_comple

### **Sweep controller checkpoints by generation metrics**

In [ ]:
# ============================================================
# Cell 15 — Train or resume ntkmirror controller
# Same training style as previous NTK notebooks
# ============================================================

import time
import random
import json
from tqdm.auto import tqdm

model.eval()

global_step = int(START_GLOBAL_STEP)
micro_step_seen = global_step * GRAD_ACCUM_STEPS

best_eval_loss = BEST_EVAL_LOSS_SO_FAR
best_model_checkpoint = BEST_CHECKPOINT_SO_FAR

running_loss = 0.0
running_items = 0

train_start = time.time()

print("Starting ntkmirror training.")
print("Starting global step:", global_step)
print("Target optimizer steps:", TOTAL_OPTIMIZER_STEPS)

optimizer.zero_grad(set_to_none=True)

stop_training = False

# Number of micro-steps across the whole planned training.
# This is needed so the final partial accumulation is stepped.
TOTAL_MICRO_STEPS = num_micro_batches_per_epoch * NUM_EPOCHS

for epoch in range(NUM_EPOCHS):
    epoch_indices = list(range(len(train_ntk_examples)))

    rng = random.Random(SEED + epoch)
    rng.shuffle(epoch_indices)

    epoch_examples = [train_ntk_examples[i] for i in epoch_indices]
    epoch_batches = list(batches(epoch_examples, PER_DEVICE_BATCH_SIZE))

    for micro_batch_index, chunk in enumerate(
        tqdm(epoch_batches, desc=f"Epoch {epoch + 1}/{NUM_EPOCHS}")
    ):
        current_micro_index = epoch * len(epoch_batches) + micro_batch_index

        # Resume support.
        if current_micro_index < micro_step_seen:
            continue

        batch = make_batch(
            tokenizer,
            chunk,
            device=device,
            max_length=MAX_SEQ_LENGTH,
        )

        if not tuner.controller.is_attached:
            tuner.controller.attach()

        loss = tuner._loss(batch)

        if NTK_L2 > 0:
            loss = loss + float(NTK_L2) * tuner.controller.s.float().pow(2).mean()

        scaled_loss = loss / GRAD_ACCUM_STEPS
        scaled_loss.backward()

        running_loss += float(loss.detach().item())
        running_items += 1

        is_accum_step = ((current_micro_index + 1) % GRAD_ACCUM_STEPS == 0)
        is_final_micro_step = ((current_micro_index + 1) >= TOTAL_MICRO_STEPS)

        should_step = is_accum_step or is_final_micro_step

        if should_step:
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

            global_step += 1

            epoch_float = epoch + (micro_batch_index + 1) / max(1, len(epoch_batches))
            current_lr = scheduler.get_last_lr()[0]

            if global_step % LOGGING_STEPS == 0 or global_step == 1:
                avg_loss = running_loss / max(1, running_items)

                row = {
                    "loss": float(avg_loss),
                    "learning_rate": float(current_lr),
                    "epoch": float(epoch_float),
                    "step": int(global_step),
                }

                log_history.append(row)

                print(
                    f"step {global_step}/{TOTAL_OPTIMIZER_STEPS} | "
                    f"epoch {epoch_float:.3f} | "
                    f"loss {avg_loss:.6f} | "
                    f"lr {current_lr:.3e}"
                )

                running_loss = 0.0
                running_items = 0

            # ------------------------------------------------
            # Eval by NLL/loss only for monitoring/resume.
            # Final checkpoint selection remains generated chrF++/spBLEU.
            # ------------------------------------------------
            if global_step % EVAL_STEPS == 0 or global_step == TOTAL_OPTIMIZER_STEPS:
                if tuner.controller.is_attached:
                    tuner.controller.remove()

                print(f"\nRunning eval NLL at step {global_step}...")

                eval_metrics = evaluate_ntk_nll(
                    eval_ntk_examples,
                    batch_size=1,
                    max_length=MAX_SEQ_LENGTH,
                    limit=EVAL_NLL_LIMIT,
                )

                eval_loss = float(eval_metrics["nll"])
                eval_token_acc = float(eval_metrics["token_acc"])

                row = {
                    "eval_loss": eval_loss,
                    "eval_token_acc": eval_token_acc,
                    "eval_tokens": float(eval_metrics["tokens"]),
                    "epoch": float(epoch_float),
                    "step": int(global_step),
                }

                log_history.append(row)

                print(
                    f"\nEVAL step {global_step}: "
                    f"eval_loss={eval_loss:.6f}, "
                    f"token_acc={eval_token_acc:.6f}, "
                    f"tokens={eval_metrics['tokens']:.0f}\n"
                )

                if best_eval_loss is None or eval_loss < float(best_eval_loss):
                    best_eval_loss = eval_loss
                    best_model_checkpoint = str(OUTPUT_DIR / f"checkpoint-{global_step}")
                    print("New best checkpoint by eval loss:", best_model_checkpoint)

                if not tuner.controller.is_attached:
                    tuner.controller.attach()

            # ------------------------------------------------
            # Save checkpoint
            # ------------------------------------------------
            if global_step % SAVE_STEPS == 0 or global_step == TOTAL_OPTIMIZER_STEPS:
                if tuner.controller.is_attached:
                    tuner.controller.remove()

                ckpt_dir = OUTPUT_DIR / f"checkpoint-{global_step}"

                save_ntk_checkpoint(
                    ckpt_dir,
                    global_step=global_step,
                    epoch=epoch_float,
                    optimizer=optimizer,
                    scheduler=scheduler,
                    log_history=log_history,
                    best_metric=best_eval_loss,
                    best_model_checkpoint=best_model_checkpoint,
                )

                print("Saved checkpoint:", ckpt_dir)

                prune_old_checkpoints(
                    OUTPUT_DIR,
                    SAVE_TOTAL_LIMIT,
                    best_model_checkpoint=best_model_checkpoint,
                )

                if not tuner.controller.is_attached:
                    tuner.controller.attach()

            if global_step >= TOTAL_OPTIMIZER_STEPS:
                stop_training = True
                break

    if stop_training:
        break

if tuner.controller is not None and tuner.controller.is_attached:
    tuner.controller.remove()

elapsed = time.time() - train_start

trainer_stats = {
    "global_step": int(global_step),
    "best_eval_loss": None if best_eval_loss is None else float(best_eval_loss),
    "best_model_checkpoint": best_model_checkpoint,
    "train_seconds": float(elapsed),
}

print("\nTraining finished.")
print(json.dumps(trainer_stats, indent=2, ensure_ascii=False))

print("\nLatest checkpoints:")
for step, ckpt in list_ntk_checkpoints(OUTPUT_DIR):
    print(f"  step={step}: {ckpt}")



NameError: name 'START_GLOBAL_STEP' is not defined

### **Save metric-best controller**

In [17]:
# ============================================================
# Cell 16 — Sweep controller checkpoints by generated metrics
# Resumable by source_id after session restart
# Select by chrF++ first, spBLEU second
# ============================================================

import re
import pandas as pd
import torch
from pathlib import Path
from sacrebleu.metrics import BLEU, CHRF
from tqdm.auto import tqdm

def clean_generated_answer(text):
    text = str(text)

    if RESPONSE_MARKER in text:
        text = text.split(RESPONSE_MARKER)[-1]

    special_tokens = [
        tokenizer.eos_token,
        tokenizer.pad_token,
        "<|endoftext|>",
        "<|im_end|>",
        "<|end|>",
    ]

    for tok in special_tokens:
        if tok:
            text = text.replace(tok, "")

    stop_markers = [
        SYSTEM_MARKER,
        INSTRUCTION_MARKER,
        RESPONSE_MARKER,
        "### System",
        "### Instruction",
        "### Arabic translation",
    ]

    for marker in stop_markers:
        if marker in text:
            text = text.split(marker)[0]

    text = re.sub(r"\s+", " ", text).strip()
    return text


def compute_lexical_metrics(preds, refs):
    bleu_metric = BLEU(tokenize="13a")
    spbleu_metric = BLEU(tokenize="flores200")
    chrf_metric = CHRF(word_order=0)
    chrfpp_metric = CHRF(word_order=2)

    return {
        "BLEU": float(bleu_metric.corpus_score(preds, [refs]).score),
        "spBLEU": float(spbleu_metric.corpus_score(preds, [refs]).score),
        "chrF": float(chrf_metric.corpus_score(preds, [refs]).score),
        "chrF++": float(chrfpp_metric.corpus_score(preds, [refs]).score),
    }


def generate_with_controller_from_row(row, max_new_tokens=MAX_NEW_TOKENS):
    user_text = make_user_prompt(row)

    prompt = format_ntkmirror_prompt(
        system_text=SYSTEM_PROMPT,
        user_text=user_text,
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
    ).to(device)

    input_len = inputs["input_ids"].shape[-1]

    model.eval()

    with torch.no_grad():
        with tuner.controller.attached():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                repetition_penalty=1.05,
                eos_token_id=tokenizer.eos_token_id,
                pad_token_id=tokenizer.pad_token_id,
            )

    generated_ids = outputs[0, input_len:]
    decoded_new = tokenizer.decode(generated_ids, skip_special_tokens=False)

    return clean_generated_answer(decoded_new)


# ------------------------------------------------------------
# Build sweep eval set
# ------------------------------------------------------------

if SWEEP_EVAL_LIMIT is None:
    sweep_eval_df = eval_df.reset_index(drop=True).copy()
else:
    sweep_eval_df = eval_df.sample(
        n=min(SWEEP_EVAL_LIMIT, len(eval_df)),
        random_state=SEED,
    ).reset_index(drop=True)

sweep_eval_df["source_id"] = sweep_eval_df["source_id"].astype(str)

print("Sweep eval examples:", len(sweep_eval_df))


# ------------------------------------------------------------
# Find checkpoints to sweep
# ------------------------------------------------------------

available_ckpts = list_ntk_checkpoints(OUTPUT_DIR)
available_by_step = {step: ckpt for step, ckpt in available_ckpts}

steps_to_sweep = [s for s in SWEEP_STEPS if s in available_by_step]

if not steps_to_sweep:
    steps_to_sweep = [step for step, _ in available_ckpts]

if not steps_to_sweep:
    raise FileNotFoundError(f"No NTK checkpoints found under {OUTPUT_DIR}")

print("Steps to sweep:", steps_to_sweep)


# ------------------------------------------------------------
# Output directory
# ------------------------------------------------------------

sweep_dir = OUTPUT_DIR / "metric_sweep"
sweep_dir.mkdir(parents=True, exist_ok=True)

sweep_rows = []


# ------------------------------------------------------------
# Sweep checkpoints
# ------------------------------------------------------------

for step in steps_to_sweep:
    ckpt_dir = available_by_step[step]
    controller_path = ckpt_dir / "controller.pt"

    pred_path = sweep_dir / f"sweep_predictions_step{step}.csv"

    print("\n==============================")
    print("Sweeping checkpoint:", ckpt_dir)
    print("Prediction file:", pred_path)
    print("==============================")

    # Make sure no old controller hook is still attached before loading.
    if getattr(tuner, "controller", None) is not None:
        try:
            if tuner.controller.is_attached:
                tuner.controller.remove()
        except Exception:
            pass

    tuner.load(controller_path, allow_model_mismatch=True)

    expected_ids = set(sweep_eval_df["source_id"].astype(str).tolist())

    # --------------------------------------------------------
    # Resume existing predictions for this checkpoint
    # --------------------------------------------------------

    if pred_path.exists():
        existing_df = pd.read_csv(pred_path)
        existing_df["source_id"] = existing_df["source_id"].astype(str)

        existing_df = existing_df[
            existing_df["source_id"].isin(expected_ids)
        ].copy()

        existing_df = existing_df.drop_duplicates(
            subset=["source_id"],
            keep="first",
        )

        existing_df["prediction"] = existing_df["prediction"].fillna("").astype(str)

        # Non-empty predictions are considered done.
        # Empty predictions are regenerated.
        done_df = existing_df[
            existing_df["prediction"].str.strip().str.len() > 0
        ].copy()

        out_rows = done_df.to_dict("records")
        done_ids = set(done_df["source_id"].astype(str).tolist())

        print(f"Resuming step {step}: {len(done_ids)} / {len(sweep_eval_df)} already done.")

    else:
        out_rows = []
        done_ids = set()
        print(f"No existing predictions for step {step}. Starting fresh.")

    # --------------------------------------------------------
    # Generate only missing rows
    # --------------------------------------------------------

    missing_eval_df = sweep_eval_df[
        ~sweep_eval_df["source_id"].astype(str).isin(done_ids)
    ].reset_index(drop=True)

    print(f"Need to generate for step {step}: {len(missing_eval_df)} remaining examples.")

    generated_since_save = 0

    for _, row in tqdm(
        missing_eval_df.iterrows(),
        total=len(missing_eval_df),
        desc=f"Generate step {step}",
    ):
        row_dict = row.to_dict()
        source_id = str(row_dict.get("source_id", ""))

        try:
            pred = generate_with_controller_from_row(row_dict)
        except Exception as e:
            pred = ""
            print("Generation failed:", source_id, repr(e))

        ref = str(row_dict["target_arabic"])

        out_rows.append({
            "source_id": source_id,
            "config": row_dict.get("config", ""),
            "domain": row_dict.get("domain", ""),
            "dialect": row_dict.get("dialect", ""),
            "source_text": row_dict.get("source_text", ""),
            "reference_arabic": ref,
            "prediction": pred,
        })

        done_ids.add(source_id)
        generated_since_save += 1

        # Save progress frequently so restart can continue.
        if generated_since_save >= 10:
            tmp_df = pd.DataFrame(out_rows)
            tmp_df["source_id"] = tmp_df["source_id"].astype(str)
            tmp_df = tmp_df.drop_duplicates(subset=["source_id"], keep="first")
            tmp_df.to_csv(pred_path, index=False, encoding="utf-8-sig")
            generated_since_save = 0

    # --------------------------------------------------------
    # Finalize prediction file for this checkpoint
    # --------------------------------------------------------

    out_df = pd.DataFrame(out_rows)

    if len(out_df) == 0:
        raise RuntimeError(f"No predictions generated or loaded for step {step}.")

    out_df["source_id"] = out_df["source_id"].astype(str)
    out_df = out_df.drop_duplicates(subset=["source_id"], keep="first")

    # Reorder exactly like sweep_eval_df.
    order_df = sweep_eval_df[["source_id"]].copy()
    order_df["source_id"] = order_df["source_id"].astype(str)

    out_df = order_df.merge(out_df, on="source_id", how="left")

    missing_count = out_df["prediction"].isna().sum()

    if missing_count > 0:
        out_df.to_csv(pred_path, index=False, encoding="utf-8-sig")
        raise RuntimeError(
            f"Step {step} is incomplete. Missing predictions: {missing_count}. "
            f"Rerun Cell 16 to continue from {pred_path}."
        )

    out_df["prediction"] = out_df["prediction"].fillna("").astype(str)
    out_df["reference_arabic"] = out_df["reference_arabic"].fillna("").astype(str)

    empty_count = (out_df["prediction"].str.strip().str.len() == 0).sum()

    if empty_count > 0:
        print(
            f"WARNING: step {step} has {empty_count} empty predictions. "
            "Metrics will count them as empty outputs."
        )

    out_df.to_csv(pred_path, index=False, encoding="utf-8-sig")

    preds = out_df["prediction"].astype(str).tolist()
    refs = out_df["reference_arabic"].astype(str).tolist()

    metrics = compute_lexical_metrics(preds, refs)

    sweep_row = {
        "step": int(step),
        "checkpoint_dir": str(ckpt_dir),
        "controller_path": str(controller_path),
        "prediction_path": str(pred_path),
        "n_examples": int(len(out_df)),
        "n_empty_predictions": int(empty_count),
        **metrics,
    }

    sweep_rows.append(sweep_row)

    print("Metrics:", sweep_row)


# ------------------------------------------------------------
# Rank checkpoints and save sweep summary
# ------------------------------------------------------------

sweep_results_df = pd.DataFrame(sweep_rows)

sweep_results_ranked_df = sweep_results_df.sort_values(
    by=[PRIMARY_SELECTION_METRIC, SECONDARY_SELECTION_METRIC, "BLEU"],
    ascending=[False, False, False],
).reset_index(drop=True)

sweep_results_path = sweep_dir / "controller_sweep_generation_metrics.csv"

sweep_results_ranked_df.to_csv(
    sweep_results_path,
    index=False,
    encoding="utf-8-sig",
)

display(sweep_results_ranked_df)

METRIC_BEST_STEP = int(sweep_results_ranked_df.iloc[0]["step"])
METRIC_BEST_CONTROLLER_PATH = Path(sweep_results_ranked_df.iloc[0]["controller_path"])

print("\nMetric-best step:", METRIC_BEST_STEP)
print("Metric-best controller:", METRIC_BEST_CONTROLLER_PATH)
print("Saved sweep results:", sweep_results_path)

Sweep eval examples: 1118
Steps to sweep: [600, 1000, 1600]

Sweeping checkpoint: /content/drive/MyDrive/alexandria_qwen35_sft/runs/qwen3_4b_lora_step700_alexandria_eg_only_context3_complete2shot_ntkmirror_PREVSTYLE_g10000_score256_layers_all_mlgate005_lr0p0005_5epochs/checkpoint-600
Prediction file: /content/drive/MyDrive/alexandria_qwen35_sft/runs/qwen3_4b_lora_step700_alexandria_eg_only_context3_complete2shot_ntkmirror_PREVSTYLE_g10000_score256_layers_all_mlgate005_lr0p0005_5epochs/metric_sweep/sweep_predictions_step600.csv
Resuming step 600: 1118 / 1118 already done.
Need to generate for step 600: 0 remaining examples.


Generate step 600: 0it [00:00, ?it/s]

Metrics: {'step': 600, 'checkpoint_dir': '/content/drive/MyDrive/alexandria_qwen35_sft/runs/qwen3_4b_lora_step700_alexandria_eg_only_context3_complete2shot_ntkmirror_PREVSTYLE_g10000_score256_layers_all_mlgate005_lr0p0005_5epochs/checkpoint-600', 'controller_path': '/content/drive/MyDrive/alexandria_qwen35_sft/runs/qwen3_4b_lora_step700_alexandria_eg_only_context3_complete2shot_ntkmirror_PREVSTYLE_g10000_score256_layers_all_mlgate005_lr0p0005_5epochs/checkpoint-600/controller.pt', 'prediction_path': '/content/drive/MyDrive/alexandria_qwen35_sft/runs/qwen3_4b_lora_step700_alexandria_eg_only_context3_complete2shot_ntkmirror_PREVSTYLE_g10000_score256_layers_all_mlgate005_lr0p0005_5epochs/metric_sweep/sweep_predictions_step600.csv', 'n_examples': 1118, 'n_empty_predictions': 0, 'BLEU': 10.130506845364634, 'spBLEU': 19.44629307955714, 'chrF': 38.05144659646341, 'chrF++': 35.26695812167313}

Sweeping checkpoint: /content/drive/MyDrive/alexandria_qwen35_sft/runs/qwen3_4b_lora_step700_alexandr

Generate step 1000: 0it [00:00, ?it/s]

Metrics: {'step': 1000, 'checkpoint_dir': '/content/drive/MyDrive/alexandria_qwen35_sft/runs/qwen3_4b_lora_step700_alexandria_eg_only_context3_complete2shot_ntkmirror_PREVSTYLE_g10000_score256_layers_all_mlgate005_lr0p0005_5epochs/checkpoint-1000', 'controller_path': '/content/drive/MyDrive/alexandria_qwen35_sft/runs/qwen3_4b_lora_step700_alexandria_eg_only_context3_complete2shot_ntkmirror_PREVSTYLE_g10000_score256_layers_all_mlgate005_lr0p0005_5epochs/checkpoint-1000/controller.pt', 'prediction_path': '/content/drive/MyDrive/alexandria_qwen35_sft/runs/qwen3_4b_lora_step700_alexandria_eg_only_context3_complete2shot_ntkmirror_PREVSTYLE_g10000_score256_layers_all_mlgate005_lr0p0005_5epochs/metric_sweep/sweep_predictions_step1000.csv', 'n_examples': 1118, 'n_empty_predictions': 0, 'BLEU': 9.970410578335246, 'spBLEU': 19.273083469210736, 'chrF': 38.00563377541318, 'chrF++': 35.17270899670489}

Sweeping checkpoint: /content/drive/MyDrive/alexandria_qwen35_sft/runs/qwen3_4b_lora_step700_alex

Generate step 1600:   0%|          | 0/608 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blo

KeyboardInterrupt: 

In [19]:
# ============================================================
# Cell 16A — Check partial metrics for one checkpoint
# Example: checkpoint-600 while Cell 16 is incomplete
# ============================================================

import pandas as pd
from pathlib import Path
from sacrebleu.metrics import BLEU, CHRF

CHECK_STEP = 1000

sweep_dir = OUTPUT_DIR / "metric_sweep"
pred_path = sweep_dir / f"sweep_predictions_step{CHECK_STEP}.csv"

if not pred_path.exists():
    raise FileNotFoundError(f"No prediction file found:\n{pred_path}")

partial_df = pd.read_csv(pred_path)
partial_df["source_id"] = partial_df["source_id"].astype(str)

# Remove duplicate source_ids if any.
partial_df = partial_df.drop_duplicates(subset=["source_id"], keep="first")

# Keep only rows with non-empty predictions.
partial_df["prediction"] = partial_df["prediction"].fillna("").astype(str)
done_df = partial_df[partial_df["prediction"].str.strip().str.len() > 0].copy()

if len(done_df) == 0:
    raise RuntimeError("No completed non-empty predictions found yet.")

# Make sure references exist.
if "reference_arabic" not in done_df.columns or done_df["reference_arabic"].isna().any():
    ref_df = eval_df[["source_id", "target_arabic"]].copy()
    ref_df["source_id"] = ref_df["source_id"].astype(str)

    done_df = done_df.drop(columns=["reference_arabic"], errors="ignore")
    done_df = done_df.merge(ref_df, on="source_id", how="left")
    done_df = done_df.rename(columns={"target_arabic": "reference_arabic"})

done_df["reference_arabic"] = done_df["reference_arabic"].fillna("").astype(str)

# Compute lexical metrics on completed predictions only.
bleu_metric = BLEU(tokenize="13a")
spbleu_metric = BLEU(tokenize="flores200")
chrf_metric = CHRF(word_order=0)
chrfpp_metric = CHRF(word_order=2)

preds = done_df["prediction"].tolist()
refs = done_df["reference_arabic"].tolist()

partial_metrics = {
    "checkpoint_step": CHECK_STEP,
    "completed_predictions": len(done_df),
    "rows_in_file": len(partial_df),
    "BLEU": float(bleu_metric.corpus_score(preds, [refs]).score),
    "spBLEU": float(spbleu_metric.corpus_score(preds, [refs]).score),
    "chrF": float(chrf_metric.corpus_score(preds, [refs]).score),
    "chrF++": float(chrfpp_metric.corpus_score(preds, [refs]).score),
}

partial_metrics_df = pd.DataFrame([partial_metrics])

display(partial_metrics_df)

print("Prediction file:")
print(pred_path)



,checkpoint_step,completed_predictions,rows_in_file,BLEU,spBLEU,chrF,chrF++
0,1000,1118,1118,9.970411,19.273083,38.005634,35.172709


Prediction file:
/content/drive/MyDrive/alexandria_qwen35_sft/runs/qwen3_4b_lora_step700_alexandria_eg_only_context3_complete2shot_ntkmirror_PREVSTYLE_g10000_score256_layers_all_mlgate005_lr0p0005_5epochs/metric_sweep/sweep_predictions_step1000.csv


In [ ]:
# ============================================================
# Cell 17 — Save metric-best controller to final_adapters
# ============================================================

import json
import shutil

if "METRIC_BEST_CONTROLLER_PATH" not in globals():
    raise RuntimeError("Run Cell 16 metric sweep first.")

CONTROLLER_DIR.mkdir(parents=True, exist_ok=True)

FINAL_CONTROLLER_PATH = CONTROLLER_DIR / f"controller_metric_best_step{METRIC_BEST_STEP}.pt"
FINAL_CONFIG_PATH = CONTROLLER_DIR / f"controller_metric_best_step{METRIC_BEST_STEP}_config.json"
FINAL_SWEEP_PATH = CONTROLLER_DIR / "controller_sweep_generation_metrics.csv"

shutil.copy2(METRIC_BEST_CONTROLLER_PATH, FINAL_CONTROLLER_PATH)

if sweep_results_path.exists():
    shutil.copy2(sweep_results_path, FINAL_SWEEP_PATH)

final_controller_config = {
    "experiment_name": EXPERIMENT_NAME,
    "base_model": MODEL_NAME,
    "lora_experiment_name": LORA_EXPERIMENT_NAME,
    "lora_adapter_path": str(LORA_ADAPTER_PATH),
    "lora_best_step": int(LORA_BEST_STEP),
    "metric_best_step": int(METRIC_BEST_STEP),
    "metric_best_controller_path": str(FINAL_CONTROLLER_PATH),
    "metric_best_source_controller_path": str(METRIC_BEST_CONTROLLER_PATH),
    "primary_selection_metric": PRIMARY_SELECTION_METRIC,
    "secondary_selection_metric": SECONDARY_SELECTION_METRIC,
    "ntk_gates": int(NTK_GATES),
    "ntk_layers": str(NTK_LAYERS),
    "ntk_max_log_gate": float(NTK_MAX_LOG_GATE),
    "ntk_hook_site": str(NTK_HOOK_SITE),
    "ntk_score_examples": int(NTK_SCORE_EXAMPLES),
    "learning_rate": float(LEARNING_RATE),
    "num_epochs": int(NUM_EPOCHS),
    "grad_accum_steps": int(GRAD_ACCUM_STEPS),
    "max_seq_length": int(MAX_SEQ_LENGTH),
}

FINAL_CONFIG_PATH.write_text(
    json.dumps(final_controller_config, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

tuner.load(FINAL_CONTROLLER_PATH, allow_model_mismatch=True)

print("Saved metric-best controller:")
print(FINAL_CONTROLLER_PATH)

print("\nSaved config:")
print(FINAL_CONFIG_PATH)

print("\nCopied sweep results:")
print(FINAL_SWEEP_PATH)

### **Full eval generation**

In [ ]:
# ============================================================
# Cell 18 — Full eval generation with metric-best controller
# Resumable by source_id
# ============================================================

import pandas as pd
from tqdm.auto import tqdm

if "FINAL_CONTROLLER_PATH" not in globals():
    raise RuntimeError("Run Cell 16 first.")

tuner.load(FINAL_CONTROLLER_PATH, allow_model_mismatch=True)

PRED_SAVE_EVERY = 50

EVAL_TAG = f"metric_best_step{METRIC_BEST_STEP}"

FULL_PRED_PATH = PRED_DIR / f"full_eval_predictions_{EXPERIMENT_NAME}_{EVAL_TAG}.csv"

print("Generating full eval predictions")
print("Experiment:", EXPERIMENT_NAME)
print("Controller:", FINAL_CONTROLLER_PATH)
print("Output:", FULL_PRED_PATH)

full_eval_df = eval_df.reset_index(drop=True).copy()
expected_ids = set(full_eval_df["source_id"].astype(str).tolist())

if FULL_PRED_PATH.exists():
    existing_df = pd.read_csv(FULL_PRED_PATH)
    existing_df["source_id"] = existing_df["source_id"].astype(str)
    existing_df = existing_df[existing_df["source_id"].isin(expected_ids)].copy()
    existing_df = existing_df.drop_duplicates(subset=["source_id"], keep="first")

    pred_rows = existing_df.to_dict("records")
    done_ids = set(existing_df["source_id"].astype(str).tolist())

    print(f"Resuming existing predictions: {len(done_ids)} / {len(full_eval_df)}")
else:
    pred_rows = []
    done_ids = set()
    print("No existing prediction file. Starting from scratch.")

for _, row in tqdm(full_eval_df.iterrows(), total=len(full_eval_df), desc="Full eval generation"):
    row_dict = row.to_dict()
    source_id = str(row_dict["source_id"])

    if source_id in done_ids:
        continue

    try:
        pred = generate_with_controller_from_row(row_dict)
    except Exception as e:
        pred = ""
        print("Generation failed:", source_id, repr(e))

    pred_rows.append({
        "source_id": source_id,
        "config": row_dict.get("config", ""),
        "split": row_dict.get("split", ""),
        "conversation_id": row_dict.get("conversation_id", ""),
        "turn_id": row_dict.get("turn_id", ""),
        "country": row_dict.get("country", ""),
        "dialect": row_dict.get("dialect", ""),
        "domain": row_dict.get("domain", ""),
        "speaker": row_dict.get("speaker", ""),
        "gender_direction": row_dict.get("gender_direction", ""),
        "source_text": row_dict.get("source_text", ""),
        "reference_arabic": row_dict.get("target_arabic", ""),
        "prediction": pred,
    })

    done_ids.add(source_id)

    if len(pred_rows) % PRED_SAVE_EVERY == 0:
        tmp_df = pd.DataFrame(pred_rows)
        tmp_df = tmp_df.drop_duplicates(subset=["source_id"], keep="first")
        tmp_df.to_csv(FULL_PRED_PATH, index=False, encoding="utf-8-sig")
        print(f"Saved progress: {len(done_ids)} / {len(full_eval_df)}")

final_pred_df = pd.DataFrame(pred_rows)
final_pred_df["source_id"] = final_pred_df["source_id"].astype(str)
final_pred_df = final_pred_df.drop_duplicates(subset=["source_id"], keep="first")

# Reorder to exactly match eval_df.
order_df = full_eval_df[["source_id"]].copy()
order_df["source_id"] = order_df["source_id"].astype(str)

final_pred_df = order_df.merge(final_pred_df, on="source_id", how="left")

final_pred_df.to_csv(FULL_PRED_PATH, index=False, encoding="utf-8-sig")

missing_predictions = final_pred_df["prediction"].isna().sum()

print("\nSaved full predictions:")
print(FULL_PRED_PATH)
print("Rows:", len(final_pred_df))
print("Missing predictions:", int(missing_predictions))

if missing_predictions == 0:
    print("Full prediction coverage confirmed.")
else:
    print("WARNING: rerun this cell to fill missing rows.")

### **Full lexical metrics**

In [ ]:
# ============================================================
# Cell 19 — Full lexical metrics: BLEU, spBLEU, chrF, chrF++
# ============================================================

import json
import pandas as pd
from sacrebleu.metrics import BLEU, CHRF

if "FULL_PRED_PATH" not in globals():
    raise RuntimeError("Run Cell 17 first.")

FULL_METRICS_PATH = PRED_DIR / f"full_eval_metrics_{EXPERIMENT_NAME}_{EVAL_TAG}.json"

pred_df = pd.read_csv(FULL_PRED_PATH)

required_cols = {"source_id", "prediction", "reference_arabic"}
missing_cols = required_cols - set(pred_df.columns)

if missing_cols:
    raise ValueError(f"Missing required prediction columns: {missing_cols}")

pred_df["source_id"] = pred_df["source_id"].astype(str)
pred_df["prediction"] = pred_df["prediction"].fillna("").astype(str)
pred_df["reference_arabic"] = pred_df["reference_arabic"].fillna("").astype(str)

expected_eval_df = eval_df.reset_index(drop=True).copy()
expected_eval_df["source_id"] = expected_eval_df["source_id"].astype(str)

expected_ids = set(expected_eval_df["source_id"])
actual_ids = set(pred_df["source_id"])

missing_ids = expected_ids - actual_ids

print("Expected examples:", len(expected_eval_df))
print("Prediction rows:", len(pred_df))
print("Missing IDs:", len(missing_ids))

if missing_ids:
    raise RuntimeError("Prediction file is incomplete. Rerun Cell 17.")

order_df = expected_eval_df[["source_id"]].copy()
pred_df_ordered = order_df.merge(pred_df, on="source_id", how="left")

preds = pred_df_ordered["prediction"].fillna("").astype(str).tolist()
refs = pred_df_ordered["reference_arabic"].fillna("").astype(str).tolist()

lexical_metrics = compute_lexical_metrics(preds, refs)

full_metrics = {
    "system": f"Qwen3-4B-LoRA-complete2shot-all-r16-best{LORA_BEST_STEP}+NTK",
    "experiment_name": EXPERIMENT_NAME,
    "base_model": MODEL_NAME,
    "lora_experiment_name": LORA_EXPERIMENT_NAME,
    "lora_adapter_path": str(LORA_ADAPTER_PATH),
    "lora_best_step": int(LORA_BEST_STEP),
    "controller_path": str(FINAL_CONTROLLER_PATH),
    "metric_best_step": int(METRIC_BEST_STEP),
    "n_examples": len(preds),
    **lexical_metrics,
}

FULL_METRICS_PATH.write_text(
    json.dumps(full_metrics, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print(json.dumps(full_metrics, indent=2, ensure_ascii=False))
print("\nSaved full metrics:")
print(FULL_METRICS_PATH)

In [ ]:
# ============================================================
# Cell 20 — Full lexical metrics: BLEU, spBLEU, chrF, chrF++
# ============================================================

import json
import pandas as pd
from sacrebleu.metrics import BLEU, CHRF

if "FULL_PRED_PATH" not in globals():
    raise RuntimeError("Run Cell 17 first.")

FULL_METRICS_PATH = PRED_DIR / f"full_eval_metrics_{EXPERIMENT_NAME}_{EVAL_TAG}.json"

pred_df = pd.read_csv(FULL_PRED_PATH)

required_cols = {"source_id", "prediction", "reference_arabic"}
missing_cols = required_cols - set(pred_df.columns)

if missing_cols:
    raise ValueError(f"Missing required prediction columns: {missing_cols}")

pred_df["source_id"] = pred_df["source_id"].astype(str)
pred_df["prediction"] = pred_df["prediction"].fillna("").astype(str)
pred_df["reference_arabic"] = pred_df["reference_arabic"].fillna("").astype(str)

expected_eval_df = eval_df.reset_index(drop=True).copy()
expected_eval_df["source_id"] = expected_eval_df["source_id"].astype(str)

expected_ids = set(expected_eval_df["source_id"])
actual_ids = set(pred_df["source_id"])

missing_ids = expected_ids - actual_ids

print("Expected examples:", len(expected_eval_df))
print("Prediction rows:", len(pred_df))
print("Missing IDs:", len(missing_ids))

if missing_ids:
    raise RuntimeError("Prediction file is incomplete. Rerun Cell 17.")

order_df = expected_eval_df[["source_id"]].copy()
pred_df_ordered = order_df.merge(pred_df, on="source_id", how="left")

preds = pred_df_ordered["prediction"].fillna("").astype(str).tolist()
refs = pred_df_ordered["reference_arabic"].fillna("").astype(str).tolist()

lexical_metrics = compute_lexical_metrics(preds, refs)

full_metrics = {
    "system": f"Qwen3-4B-LoRA-complete2shot-all-r16-best{LORA_BEST_STEP}+NTK",
    "experiment_name": EXPERIMENT_NAME,
    "base_model": MODEL_NAME,
    "lora_experiment_name": LORA_EXPERIMENT_NAME,
    "lora_adapter_path": str(LORA_ADAPTER_PATH),
    "lora_best_step": int(LORA_BEST_STEP),
    "controller_path": str(FINAL_CONTROLLER_PATH),
    "metric_best_step": int(METRIC_BEST_STEP),
    "n_examples": len(preds),
    **lexical_metrics,
}

FULL_METRICS_PATH.write_text(
    json.dumps(full_metrics, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print(json.dumps(full_metrics, indent=2, ensure_ascii=False))
print("\nSaved full metrics:")
print(FULL_METRICS_PATH)